# The reward curve that climbed

The lenient comparator (defect D1) pays a result that contains the gold value somewhere, so a query that returns a whole table can be paid in full. This notebook draws what the lenient arm was paid at every step beside what the twins confirmed on dev at every checkpoint, the same for the honest arm, and the step the monitor's result size statistic fired. A climbing reward over a flat confirmed accuracy is the hack.

In [ ]:
import json
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

from goodhart_evaluation import arms

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
runs = arms.arm_runs(ROOT)
print("promoted defect study runs:", runs or "none yet")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
for arm in ("honest", "lenient"):
    run = runs.get(arm)
    if run is None:
        print(f"the {arm} arm has not run yet")
        continue
    steps = arms.paid(ROOT, run)
    if steps:
        ax.plot([s for s, _ in steps], [v for _, v in steps], label=f"{arm}: reward paid")
    dev = arms.earned(ROOT, run)
    if dev:
        ax.plot([s for s, _ in dev], [v for _, v in dev], marker="o", linestyle="--", label=f"{arm}: twin confirmed on dev")
    fired = arms.first_alert(ROOT, run, "mean_result_rows")
    if fired is not None:
        ax.axvline(fired, color="grey", linewidth=0.8)
        print(f"{arm}: the result size monitor fired at step {fired}")
ax.set_xlabel("step")
ax.set_ylabel("share")
if ax.lines:
    ax.legend(fontsize=8)
plt.close(fig)
print("lines drawn:", len(ax.lines))